In [1]:
# A100 train
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [5]:
ds['train'][:3]

{'output': ['1. 这一天阳光明媚，温暖舒适，与往常的冰冷酷寒形成了鲜明的对比。\n2. 这一天空气清新，没有丝毫的污染，让人感受到了大自然的洗礼。\n3. 这一天人们精神焕发，欢笑迭起，整个城市都弥漫着愉快的气氛。',
  '2磅等于0.907公斤。',
  '正在为您播放披头士的“Yesterday”这首歌。'],
 'input': ['', '', '输出：（播放“Yesterday”这首歌）'],
 'instruction': ['生成三个句子，描述这一天的不同特点。',
  '将2磅转换为千克。',
  '输入：我想听披头士的“Yesterday”这首歌。']}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("/tmp/code/huggingface/hub/models--Langboat--bloom-1b4-zh/snapshots/cd76063fd0cde2b9bcc02901fcabd35ba71eff39")
tokenizer

BloomTokenizerFast(name_or_path='/tmp/code/huggingface/hub/models--Langboat--bloom-1b4-zh/snapshots/cd76063fd0cde2b9bcc02901fcabd35ba71eff39', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:02<00:00, 2287.15 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'Human: 将2磅转换为千克。\n\nAssistant: 2磅等于0.907公斤。</s>'

In [10]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

'2磅等于0.907公斤。</s>'

In [11]:
model = AutoModelForCausalLM.from_pretrained("/tmp/code/huggingface/hub/models--Langboat--bloom-1b4-zh/snapshots/cd76063fd0cde2b9bcc02901fcabd35ba71eff39")

In [12]:
from peft import PrefixTuningConfig, get_peft_model, TaskType

config = PrefixTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=10, prefix_projection=True)
config

PrefixTuningConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.PREFIX_TUNING: 'PREFIX_TUNING'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, num_virtual_tokens=10, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, encoder_hidden_size=None, prefix_projection=True)

In [13]:
model = get_peft_model(model, config)

In [14]:
model.print_trainable_parameters()

trainable params: 205,641,728 || all params: 1,508,753,408 || trainable%: 13.6299


In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy='steps',
    logging_steps=50,
    num_train_epochs=1
)

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

/tmp/ipykernel_267/585344873.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-12-09 10:59:06,756] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: 没有那个文件或目录


[2025-12-09 10:59:09,276] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [17]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.588100,2.435181
100,2.322100,2.332658
150,2.268700,2.292880
200,2.260300,2.262246
250,2.256500,2.247879
300,2.233600,2.231820
350,2.200900,2.223167
400,2.217300,2.215238
450,2.188800,2.209909
500,2.216200,2.200307


Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_de

TrainOutput(global_step=672, training_loss=2.2548188879376365, metrics={'train_runtime': 2962.7788, 'train_samples_per_second': 7.252, 'train_steps_per_second': 0.227, 'total_flos': 2.140128746815488e+16, 'train_loss': 2.2548188879376365, 'epoch': 1.0})

In [18]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 在面对考试时，考生需要掌握一些技巧来提高成绩和通过考试。下面是一些建议:\n\n1. 提前准备： 提前一天或一周开始学习，并且做好功课。这将有助于提高考试时的复习和考试前的准备。你可以从书籍、网络资源、试卷、练习题和参考答案中获取有用的信息。这将有助于提高你的信心和完成考试的品质。\n\n2. 保持安静： 考试时保持安静，专心听课。安静的听课会让你感觉更清晰并提高学习效率'